In [1]:
 # 03_s5_diffusion_prep
#
# Stream 5 prep notebook — simple fixed-path version.
#
# This version uses the known project files directly and rebuilds the
# half-hourly `spike_start` / `neg_start` labels from `master.parquet`
# using the same rule as notebook 02b.
#
# Outputs:
# - `returns_s5.parquet`
# - `norm_stats_s5.json`
# - `conditioning_stats_s5.json`
# - `jump_labels_s5.json`

In [2]:
# If needed in a fresh Colab runtime:
# %pip install -q pyarrow pandas numpy

import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
np.random.seed(42)

Mounted at /content/drive


In [3]:
# ------------------------------------------------------------------
# Fixed project paths
# ------------------------------------------------------------------

DATA_DIR = Path("/content/drive/MyDrive/energy_synthetic_data/data")

MASTER_PATH = DATA_DIR / "master.parquet"
SEGMENTS_PATH = DATA_DIR / "segments_s2.parquet"
HAZARD_SEG_PATH = DATA_DIR / "hazard_features_seg.parquet"
FEATURE_ANALYSIS_PATH = DATA_DIR / "feature_analysis_s2.json"

OUT_RETURNS = DATA_DIR / "returns_s5.parquet"
OUT_NORM = DATA_DIR / "norm_stats_s5.json"
OUT_COND = DATA_DIR / "conditioning_stats_s5.json"
OUT_JUMPS = DATA_DIR / "jump_labels_s5.json"

for p in [MASTER_PATH, SEGMENTS_PATH, HAZARD_SEG_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

print("Using:")
print(" ", MASTER_PATH)
print(" ", SEGMENTS_PATH)
print(" ", HAZARD_SEG_PATH)

Using:
  /content/drive/MyDrive/energy_synthetic_data/data/master.parquet
  /content/drive/MyDrive/energy_synthetic_data/data/segments_s2.parquet
  /content/drive/MyDrive/energy_synthetic_data/data/hazard_features_seg.parquet


In [4]:
# ------------------------------------------------------------------
# Stream 5 config
# ------------------------------------------------------------------

SEG_LEN = 72
STEP_SIZE = 24
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
EMBARGO_HALFHOURS = 72

# GB power can go negative, so do NOT use log(price) or log-returns here.
TARGET_KIND = "asinh_diff"   # "diff" or "asinh_diff"
TAIL_Q = 0.98
BROADEN_FORWARD_ONE_STEP = True

PRICE_COL = "mid_price"

ECON_COLS = [
    "nuclear_prev_mean",
    "wind_prev_mean",
    "RV_prev",
    "ccgt_fraction_prev_mean",
    "ocgt_active_prev_mean",
    "wind_var_prev_mean",
    "har_rv_prev_mean",
]

REGIME_COL = "regime_refined"

HAZ_SEG_COLS = [
    "seg_lambda_spike_mean",
    "seg_lambda_neg_mean",
    "seg_lambda_spike_max",
]

In [5]:
# ------------------------------------------------------------------
# Load upstream data
# ------------------------------------------------------------------

master = pd.read_parquet(MASTER_PATH).copy()
segments = pd.read_parquet(SEGMENTS_PATH).copy()
hazard_seg = pd.read_parquet(HAZARD_SEG_PATH).copy()

feature_analysis = {}
if FEATURE_ANALYSIS_PATH.exists():
    with open(FEATURE_ANALYSIS_PATH, "r") as f:
        feature_analysis = json.load(f)

master["start_time"] = pd.to_datetime(master["start_time"], utc=True, errors="coerce")
master = master.sort_values("start_time").reset_index(drop=True)
segments = segments.sort_values("start_idx").reset_index(drop=True)
hazard_seg = hazard_seg.sort_values("start_idx").reset_index(drop=True)

if PRICE_COL not in master.columns:
    raise KeyError(f"{PRICE_COL=} not found in master.parquet")

for c in ECON_COLS:
    if c not in segments.columns:
        raise KeyError(f"Missing econometric conditioning column in segments_s2.parquet: {c}")

if REGIME_COL not in segments.columns:
    raise KeyError(f"Missing {REGIME_COL} in segments_s2.parquet")

for c in HAZ_SEG_COLS:
    if c not in hazard_seg.columns:
        raise KeyError(f"Missing hazard segment column in hazard_features_seg.parquet: {c}")

# Merge segment hazard features onto the segment table.
# The earlier broken version checked hazard_features_seg.parquet but forgot to merge it.
haz_small = hazard_seg[["start_idx"] + HAZ_SEG_COLS].drop_duplicates("start_idx")
segments = segments.drop(columns=[c for c in HAZ_SEG_COLS if c in segments.columns], errors="ignore")
segments = segments.merge(haz_small, on="start_idx", how="left", validate="1:1")

print("Shapes:")
print("  master    :", master.shape)
print("  segments  :", segments.shape)
print("  hazard_seg:", hazard_seg.shape)

Shapes:
  master    : (157776, 96)
  segments  : (6530, 78)
  hazard_seg: (6530, 5)


In [6]:
# ------------------------------------------------------------------
# Recompute split defensively from segment order
# ------------------------------------------------------------------

def blocked_segment_split(n_rows, train_frac=0.70, val_frac=0.15, embargo_halfhours=72, step_size=24):
    emb_rows = int(np.ceil(embargo_halfhours / step_size))
    train_end = int(np.floor(n_rows * train_frac))
    val_end = int(np.floor(n_rows * (train_frac + val_frac)))

    split = np.full(n_rows, "embargo", dtype=object)

    train_stop = max(0, train_end - emb_rows)
    val_start = min(n_rows, train_end + emb_rows)
    val_stop = max(val_start, val_end - emb_rows)
    test_start = min(n_rows, val_end + emb_rows)

    split[:train_stop] = "train"
    split[val_start:val_stop] = "val"
    split[test_start:] = "test"
    return split

segments["split"] = blocked_segment_split(
    len(segments),
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    embargo_halfhours=EMBARGO_HALFHOURS,
    step_size=STEP_SIZE,
)

print(segments["split"].value_counts(dropna=False).to_dict())

{'train': 4568, 'test': 977, 'val': 973, 'embargo': 12}


In [7]:
# ------------------------------------------------------------------
# Rebuild 02b-style half-hourly event labels from master.parquet
# ------------------------------------------------------------------

master["raw_diff"] = pd.to_numeric(master[PRICE_COL], errors="coerce").diff()
master["abs_ret"] = master["raw_diff"].abs()

ROLL_WIN = 48 * 30  # 30 days of half-hourly data

master["roll_med"] = (
    pd.to_numeric(master[PRICE_COL], errors="coerce")
    .rolling(ROLL_WIN, min_periods=48 * 7, center=True)
    .median()
)

master["roll_mad"] = (
    pd.to_numeric(master[PRICE_COL], errors="coerce")
    .rolling(ROLL_WIN, min_periods=48 * 7, center=True)
    .apply(lambda x: np.median(np.abs(x - np.median(x))), raw=True)
)

price = pd.to_numeric(master[PRICE_COL], errors="coerce")
roll_med = pd.to_numeric(master["roll_med"], errors="coerce")
roll_mad = pd.to_numeric(master["roll_mad"], errors="coerce")
abs_ret = pd.to_numeric(master["abs_ret"], errors="coerce")

# Exact 02b priority order:
# negative first, then spike, then volatile, else calm
is_neg = price < -20
is_spike = (price > (roll_med + 5.0 * roll_mad)) | (price > 300.0)
is_volatile = abs_ret > (3.0 * roll_mad)

regime_hh = np.where(
    is_neg,
    "negative",
    np.where(
        is_spike,
        "spike",
        np.where(is_volatile, "volatile", "calm"),
    ),
)

master["regime_hh"] = regime_hh
master["spike_start"] = ((master["regime_hh"] == "spike") & (master["regime_hh"].shift(1).fillna("calm") != "spike")).astype(np.int8)
master["neg_start"] = ((master["regime_hh"] == "negative") & (master["regime_hh"].shift(1).fillna("calm") != "negative")).astype(np.int8)

print("Half-hourly event counts rebuilt from master:")
print("  spike_start:", int(master["spike_start"].sum()))
print("  neg_start  :", int(master["neg_start"].sum()))

Half-hourly event counts rebuilt from master:
  spike_start: 2389
  neg_start  : 594


In [8]:
# ------------------------------------------------------------------
# Target transform and train-split normalisation
# ------------------------------------------------------------------

# Build a train-bin mask from train segments.
train_bin_mask = np.zeros(len(master), dtype=bool)
for _, seg in segments.loc[segments["split"].eq("train")].iterrows():
    s = int(seg["start_idx"])
    e = min(int(seg["start_idx"]) + SEG_LEN, len(master))
    train_bin_mask[s:e] = True

train_raw = master.loc[train_bin_mask, "raw_diff"].to_numpy(dtype=float)
train_raw = train_raw[np.isfinite(train_raw)]

if len(train_raw) == 0:
    raise RuntimeError("No finite train raw_diff values found.")

raw_tail_threshold = float(np.quantile(np.abs(train_raw), TAIL_Q))

if TARGET_KIND == "diff":
    master["target_raw"] = master["raw_diff"].astype(float)
    asinh_scale = None
elif TARGET_KIND == "asinh_diff":
    asinh_scale = float(np.nanmedian(np.abs(train_raw)))
    if (not np.isfinite(asinh_scale)) or (asinh_scale <= 1e-8):
        asinh_scale = float(np.nanstd(train_raw))
    if (not np.isfinite(asinh_scale)) or (asinh_scale <= 1e-8):
        asinh_scale = 1.0
    master["target_raw"] = np.arcsinh(master["raw_diff"].astype(float) / asinh_scale)
else:
    raise ValueError("TARGET_KIND must be 'diff' or 'asinh_diff'")

train_target = master.loc[train_bin_mask, "target_raw"].to_numpy(dtype=float)
train_target = train_target[np.isfinite(train_target)]

target_mean = float(np.nanmean(train_target))
target_std = float(np.nanstd(train_target))
target_std = max(target_std, 1e-8)

master["target_z"] = (master["target_raw"] - target_mean) / target_std

print("Target kind:", TARGET_KIND)
print("asinh_scale:", asinh_scale if TARGET_KIND == "asinh_diff" else None)
print("Train |diff| q98:", raw_tail_threshold)

Target kind: asinh_diff
asinh_scale: 2.799999999999997
Train |diff| q98: 148.0


In [9]:
# ------------------------------------------------------------------
# Conditioning preparation
# ------------------------------------------------------------------

# First, inspect the seven econometric scalar columns on the TRAIN split.
train_mask_seg = segments["split"].eq("train").to_numpy()

train_nonnull = (
    segments.loc[train_mask_seg, ECON_COLS]
    .apply(lambda s: pd.to_numeric(s, errors="coerce").notna().sum())
    .rename("train_non_null")
    .to_frame()
)
train_nonnull["train_total"] = int(train_mask_seg.sum())
print("Train non-null counts for econometric scalars:")
print(train_nonnull)

dead_econ_cols = train_nonnull.index[train_nonnull["train_non_null"].eq(0)].tolist()
print("Dead econometric cols on train:", dead_econ_cols)

# If HAR is dead, restore it from the 02b logic on the master grid, then
# aggregate back to segments using the same segment window.
if "har_rv_prev_mean" in dead_econ_cols:
    print("\nRestoring har_rv_prev_mean from 02b logic...")

    DAY_BINS = 48
    WEEK_BINS = 48 * 7
    MONTH_BINS = 48 * 30

    if "returns" in master.columns:
        master_ret = pd.to_numeric(master["returns"], errors="coerce")
    else:
        # Fallback if the explicit returns column is unavailable.
        master_ret = pd.to_numeric(master["raw_diff"], errors="coerce")

    master["returns_for_har"] = master_ret
    master["returns_sq_for_har"] = master["returns_for_har"].fillna(0.0) ** 2

    master["har_rv_prev_1d"] = (
        master["returns_sq_for_har"]
        .rolling(DAY_BINS, min_periods=24)
        .sum()
        .shift(1)
    )
    master["har_rv_prev_1w"] = (
        master["har_rv_prev_1d"]
        .rolling(WEEK_BINS, min_periods=DAY_BINS * 3)
        .mean()
    )
    master["har_rv_prev_1m"] = (
        master["har_rv_prev_1d"]
        .rolling(MONTH_BINS, min_periods=WEEK_BINS)
        .mean()
    )
    master["har_rv_prev_mean_restored"] = (
        master["har_rv_prev_1d"]
        + master["har_rv_prev_1w"]
        + master["har_rv_prev_1m"]
    ) / 3.0

    restored_vals = []
    for _, seg in segments.iterrows():
        s = int(seg["start_idx"])
        e = min(s + SEG_LEN, len(master))
        restored_vals.append(float(np.nanmean(master["har_rv_prev_mean_restored"].iloc[s:e])))

    segments["har_rv_prev_mean"] = restored_vals

    # Re-check.
    train_nonnull = (
        segments.loc[train_mask_seg, ECON_COLS]
        .apply(lambda s: pd.to_numeric(s, errors="coerce").notna().sum())
        .rename("train_non_null")
        .to_frame()
    )
    train_nonnull["train_total"] = int(train_mask_seg.sum())
    print("\nAfter HAR restore:")
    print(train_nonnull)

    dead_econ_cols = train_nonnull.index[train_nonnull["train_non_null"].eq(0)].tolist()
    print("Dead econometric cols after restore:", dead_econ_cols)

# Last-resort fallback: if HAR is still dead, drop it and go temporarily to 18 dims.
if "har_rv_prev_mean" in dead_econ_cols:
    print("\nHAR is still dead after restore attempt -> dropping it temporarily and using 18-d conditioning.")
    ECON_COLS = [c for c in ECON_COLS if c != "har_rv_prev_mean"]

# Regime one-hot order: prefer the saved order if present.
regime_refined_dist = feature_analysis.get("regime_refined_distribution", None)
if isinstance(regime_refined_dist, dict) and len(regime_refined_dist) > 0:
    regime_levels = list(regime_refined_dist.keys())
else:
    regime_levels = sorted(segments[REGIME_COL].dropna().astype(str).unique().tolist())

if len(regime_levels) != 9:
    print(f"Warning: expected 9 refined regime levels, found {len(regime_levels)}")

print("\nRegime levels:", regime_levels)

# Z-score econometric block on train split.
econ_stats = {}
for c in ECON_COLS:
    x_train = pd.to_numeric(segments.loc[train_mask_seg, c], errors="coerce")
    mu = float(np.nanmean(x_train))
    sd = float(np.nanstd(x_train))
    sd = max(sd, 1e-8)
    segments[c] = (pd.to_numeric(segments[c], errors="coerce") - mu) / sd
    econ_stats[c] = {"mean": mu, "std": sd}

# One-hot refined regime block.
for lvl in regime_levels:
    segments[f"reg_{lvl}"] = (segments[REGIME_COL].astype(str) == str(lvl)).astype(float)

# log1p + z-score hazard block on train split.
haz_stats = {}
for c in HAZ_SEG_COLS:
    x = np.log1p(pd.to_numeric(segments[c], errors="coerce").clip(lower=0))
    mu = float(np.nanmean(x[train_mask_seg]))
    sd = float(np.nanstd(x[train_mask_seg]))
    sd = max(sd, 1e-8)
    segments[f"{c}_z"] = (x - mu) / sd
    haz_stats[c] = {"mean_log1p": mu, "std_log1p": sd}


Train non-null counts for econometric scalars:
                         train_non_null  train_total
nuclear_prev_mean                  4568         4568
wind_prev_mean                     4568         4568
RV_prev                            4568         4568
ccgt_fraction_prev_mean            4567         4568
ocgt_active_prev_mean              4567         4568
wind_var_prev_mean                 4568         4568
har_rv_prev_mean                      0         4568
Dead econometric cols on train: ['har_rv_prev_mean']

Restoring har_rv_prev_mean from 02b logic...


/tmp/ipykernel_2569/3684016372.py:65: RuntimeWarning: Mean of empty slice
  restored_vals.append(float(np.nanmean(master["har_rv_prev_mean_restored"].iloc[s:e])))



After HAR restore:
                         train_non_null  train_total
nuclear_prev_mean                  4568         4568
wind_prev_mean                     4568         4568
RV_prev                            4568         4568
ccgt_fraction_prev_mean            4567         4568
ocgt_active_prev_mean              4567         4568
wind_var_prev_mean                 4568         4568
har_rv_prev_mean                   4557         4568
Dead econometric cols after restore: []

Regime levels: ['volatile', 'negative', 'spike_moderate_sustained', 'calm_mid_wind_calm', 'spike_extreme_sustained', 'calm_high_wind_calm', 'calm_low_wind_calm', 'spike_moderate_brief', 'spike_extreme_brief']


In [10]:
# ------------------------------------------------------------------
# Build one row per 72-step segment
# ------------------------------------------------------------------

rows = []
jump_summary_rows = []

for i, seg in segments.iterrows():
    if str(seg["split"]) == "embargo":
        continue

    s = int(seg["start_idx"])
    e = s + SEG_LEN
    if e > len(master):
        continue

    target_z = master["target_z"].iloc[s:e].to_numpy(dtype=float)
    raw_diff = master["raw_diff"].iloc[s:e].to_numpy(dtype=float)
    price_path = pd.to_numeric(master[PRICE_COL].iloc[s:e], errors="coerce").to_numpy(dtype=float)
    spike = master["spike_start"].iloc[s:e].to_numpy(dtype=int)
    neg = master["neg_start"].iloc[s:e].to_numpy(dtype=int)

    if len(target_z) != SEG_LEN or len(raw_diff) != SEG_LEN:
        continue

    jump_mask = ((spike == 1) | (neg == 1) | (np.abs(raw_diff) >= raw_tail_threshold)).astype(np.int8)
    if BROADEN_FORWARD_ONE_STEP:
        jump_mask[1:] = np.maximum(jump_mask[1:], jump_mask[:-1])

    jump_size = np.where(jump_mask == 1, target_z, 0.0)

    rec = {
        "seg_id": int(seg["seg_id"]) if "seg_id" in segments.columns else int(i),
        "start_idx": s,
        "split": str(seg["split"]),
        "p0_raw": float(price_path[0]),
        "price_col": PRICE_COL,
        "target_kind": TARGET_KIND,
    }

    for t in range(SEG_LEN):
        rec[f"r_{t:03d}"] = float(target_z[t])
        rec[f"jump_mask_{t:03d}"] = int(jump_mask[t])
        rec[f"jump_size_{t:03d}"] = float(jump_size[t])

    # cond_000 .. cond_006
    for j, c in enumerate(ECON_COLS):
        rec[f"cond_{j:03d}"] = float(seg[c])

    # cond_007 .. cond_015
    regime_value = str(seg[REGIME_COL])
    for j, level in enumerate(regime_levels):
        rec[f"cond_{7 + j:03d}"] = float(regime_value == level)

    # cond_016 .. cond_018
    for j, c in enumerate(HAZ_SEG_COLS):
        rec[f"cond_{16 + j:03d}"] = float(seg[c])

    rows.append(rec)

    jump_summary_rows.append({
        "split": str(seg["split"]),
        "seg_id": rec["seg_id"],
        "start_idx": s,
        "jump_share": float(jump_mask.mean()),
        "n_spike_start": int(spike.sum()),
        "n_neg_start": int(neg.sum()),
        "tail_hits": int((np.abs(raw_diff) >= raw_tail_threshold).sum()),
    })

returns_s5 = pd.DataFrame(rows)
jump_summary = pd.DataFrame(jump_summary_rows)

print("returns_s5 shape:", returns_s5.shape)
print("jump summary by split:")
if len(jump_summary):
    print(jump_summary.groupby("split")["jump_share"].describe())
else:
    print("No rows built.")

returns_s5 shape: (6518, 241)
jump summary by split:
        count      mean       std  min  25%       50%       75%       max
split                                                                    
test    977.0  0.012510  0.029521  0.0  0.0  0.000000  0.000000  0.277778
train  4568.0  0.069104  0.100581  0.0  0.0  0.027778  0.111111  0.638889
val     973.0  0.029005  0.047486  0.0  0.0  0.000000  0.041667  0.277778


In [11]:
# ------------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------------

returns_s5.to_parquet(OUT_RETURNS, index=False)

norm_stats = {
    "target_kind": TARGET_KIND,
    "price_col": PRICE_COL,
    "segment_length": SEG_LEN,
    "step_size": STEP_SIZE,
    "target_mean": target_mean,
    "target_std": target_std,
    "raw_tail_threshold_q": TAIL_Q,
    "raw_tail_threshold": raw_tail_threshold,
    "asinh_scale": asinh_scale,
}

conditioning_stats = {
    "conditioning_dim": len(ECON_COLS) + len(regime_levels) + len(HAZ_SEG_COLS),
    "econometric_cols": ECON_COLS,
    "regime_col": REGIME_COL,
    "regime_levels": regime_levels,
    "hazard_segment_cols": HAZ_SEG_COLS,
    "econometric_stats": econ_stats,
    "hazard_stats": haz_stats,
}

jump_label_stats = {
    "base_event_rule": "spike_start OR neg_start rebuilt from 02b price-priority logic on master.parquet",
    "tail_rule_q": TAIL_Q,
    "tail_rule_threshold_abs_raw_diff": raw_tail_threshold,
    "broaden_forward_one_step": BROADEN_FORWARD_ONE_STEP,
    "halfhour_event_counts": {
        "spike_start": int(master["spike_start"].sum()),
        "neg_start": int(master["neg_start"].sum()),
    },
    "segment_jump_share_by_split": (
        jump_summary.groupby("split")["jump_share"].mean().to_dict() if len(jump_summary) else {}
    ),
}

with open(OUT_NORM, "w") as f:
    json.dump(norm_stats, f, indent=2)

with open(OUT_COND, "w") as f:
    json.dump(conditioning_stats, f, indent=2)

with open(OUT_JUMPS, "w") as f:
    json.dump(jump_label_stats, f, indent=2)

print("\nSaved:")
print(" ", OUT_RETURNS)
print(" ", OUT_NORM)
print(" ", OUT_COND)
print(" ", OUT_JUMPS)


Saved:
  /content/drive/MyDrive/energy_synthetic_data/data/returns_s5.parquet
  /content/drive/MyDrive/energy_synthetic_data/data/norm_stats_s5.json
  /content/drive/MyDrive/energy_synthetic_data/data/conditioning_stats_s5.json
  /content/drive/MyDrive/energy_synthetic_data/data/jump_labels_s5.json


In [12]:
print("\nDone.")
print("Rows written:", len(returns_s5))
print("Conditioning dim:", len(ECON_COLS) + len(regime_levels) + len(HAZ_SEG_COLS))
print("Regime levels:", len(regime_levels))


Done.
Rows written: 6518
Conditioning dim: 19
Regime levels: 9
